# DATA CLEANING PART

### Column removal

In [ ]:
import pandas as pd
import os

def process_csv(source_file, target_dir, output_name, columns_to_drop):
    # Load
    df = pd.read_csv(source_file)
    
    # Drop columns
    df.drop(columns=columns_to_drop, inplace=True)
    
    # Create target directory if needed
    os.makedirs(target_dir, exist_ok=True)
    
    # Save
    output_path = os.path.join(target_dir, output_name)
    df.to_csv(output_path, index=False)
    
    return output_path

#process_csv("artist_reactions.csv", "processed/", "artist_reactions.csv", ["reacted_at"])
#process_csv("artists.csv", "processed/", "artists.csv", ["cover_id", "name", "description", "score", "rank", "metadata", "created_at", "updated_at"])
#process_csv("track_reactions.csv", "processed/", "track_reactions.csv", ["reacted_at", "genre_id"])
#process_csv("tracks.csv", "processed/", "tracks.csv", ["file_type", "mime_type", "extension", "performer", "title", "duration", "cover_id", "album_id", "score", "rank", "metadata", "created_at",
                                                       #"updated_at"])
#process_csv("user_musicbot_state.csv", "processed/", "user_musicbot_state.csv", ["cover_id", "description", "score", "rank", "metadata", "updated_at", "created_at"])
#process_csv("reaction_types.csv", "processed/", "reaction_types.csv", ["description"])

'processed/tracks.csv'

In [32]:
import pandas as pd

def remove_brackets(filepath, columns, save=False):
    """
    Remove exactly one leading '{' and one trailing '}' from each value in specified columns.
    
    Parameters:
    filepath : str - path to the CSV file.
    columns : list - column names to clean.
    save : bool (default False) - if True, overwrite the original file.
    
    Returns:
    pandas.DataFrame - cleaned DataFrame.
    """
    df = pd.read_csv(filepath)
    
    for col in columns:
        if col not in df.columns:
            print(f"Warning: Column '{col}' not found – skipping.")
            continue
        
        # Apply the transformation to each cell
        df[col] = df[col].astype(str).apply(
            lambda s: s[1:-1] if s.startswith('{') and s.endswith('}') else s
        )
    
    if save:
        df.to_csv(filepath, index=False)
        print(f"Changes saved to {filepath}")
    
    return df

cleaned = remove_brackets("processed/tracks.csv", ["artists_id", "uploaded_by"], True)

Changes saved to processed/tracks.csv


### Checking for NAN values

In [33]:
import pandas as pd

def check_missing(filepath):
    """
    Load a CSV, treat common placeholders as missing, and report counts per column.
    
    Parameters:
    filepath : str - path to CSV file.
    
    Returns:
    pandas.Series - counts of missing values per column.
    """
    # Common placeholder values that should be treated as missing
    placeholders = ['', 'N/A', 'NULL', 'null', ' ', 'NaN', 'nan', 'NA', 'na']
    
    # Load with na_values so these become NaN
    df = pd.read_csv(filepath, na_values=placeholders)
    
    # Count missing per column
    missing_counts = df.isna().sum()
    
    print(f"Checking: {filepath}")
    if missing_counts.sum() == 0:
        print("No missing values found.\n")
    else:
        print("Missing value counts per column (placeholders treated as NaN):")
        print(missing_counts[missing_counts > 0])
        print()
    
    return missing_counts

counts = check_missing("processed/artist_reactions.csv")
counts = check_missing("processed/artists.csv")
counts = check_missing("processed/reaction_types.csv")
counts = check_missing("processed/track_reactions.csv")
counts = check_missing("processed/tracks.csv")
counts = check_missing("processed/user_musicbot_state.csv")

Checking: processed/artist_reactions.csv
Missing value counts per column (placeholders treated as NaN):
on_user_id    208
dtype: int64

Checking: processed/artists.csv
No missing values found.

Checking: processed/reaction_types.csv
No missing values found.

Checking: processed/track_reactions.csv
Missing value counts per column (placeholders treated as NaN):
on_user_id    256
dtype: int64

Checking: processed/tracks.csv
Missing value counts per column (placeholders treated as NaN):
artists_id     467
uploaded_by     77
dtype: int64

Checking: processed/user_musicbot_state.csv
No missing values found.



### Some further checkings

In [41]:
import pandas as pd

# Load the CSV files
reactions = pd.read_csv("processed/artist_reactions.csv")
artists = pd.read_csv("processed/artists.csv")
tracks = pd.read_csv("processed/tracks.csv")

# ----- Extract sets of artist IDs -----

# 1. From reactions: column 'artist_id' (single values)
reactions_ids = set(reactions['artist_id'].dropna().astype(int))

# 2. From artists: column 'id' (single values)
artists_ids = set(artists['id'].dropna().astype(int))

# 3. From tracks: column 'artist_id' contains comma‑separated integers
# Split on ',', strip whitespace, drop empty strings, convert to int
track_ids_series = tracks['artists_id'].dropna().str.split(',')
# Explode into a long series, strip, convert to int
track_ids = (
    track_ids_series
    .explode()
    .str.strip()
    .loc[lambda x: x != '']          # remove any empty strings
    .astype(int)
    .unique()
)
track_ids_set = set(track_ids)

# ----- Compare the three sets -----

print("Number of unique artist IDs:")
print(f"  reactions: {len(reactions_ids)}")
print(f"  artists:   {len(artists_ids)}")
print(f"  tracks:    {len(track_ids_set)}")
print()

# Check if all three sets are equal
if reactions_ids == artists_ids == track_ids_set:
    print("✅ All three sets are identical.")
else:
    # Find differences using set operations
    only_in_reactions = reactions_ids - artists_ids - track_ids_set
    only_in_artists = artists_ids - reactions_ids - track_ids_set
    only_in_tracks = track_ids_set - reactions_ids - artists_ids

    # Also check pairs
    if only_in_reactions:
        print(f"\n  IDs present only in reactions: {sorted(only_in_reactions)}")
    if only_in_artists:
        print(f"\n  IDs present only in artists:   {sorted(only_in_artists)}")
    if only_in_tracks:
        print(f"\n  IDs present only in tracks:    {sorted(only_in_tracks)}")

    # Optional: show IDs missing from one source but present in the other two
    missing_from_artists = (reactions_ids | track_ids_set) - artists_ids
    missing_from_reactions = (artists_ids | track_ids_set) - reactions_ids
    missing_from_tracks = (reactions_ids | artists_ids) - track_ids_set

    if artists_ids == track_ids_set:
        print("✅ artists and tracks have identical ID sets.")
    else:
        print("❌ artists and tracks differ:")
        print("  Only in artists:", artists_ids - track_ids_set)
        print("  Only in tracks:", track_ids_set - artists_ids)

    print("❌ The sets are not identical. Differences:")

    if missing_from_artists:
        print(f"\n  IDs in reactions or tracks but missing from artists: {sorted(missing_from_artists)}")
    if missing_from_reactions:
        print(f"\n  IDs in artists or tracks but missing from reactions: {sorted(missing_from_reactions)}")
    if missing_from_tracks:
        print(f"\n  IDs in reactions or artists but missing from tracks:  {sorted(missing_from_tracks)}")

Number of unique artist IDs:
  reactions: 4284
  artists:   5234
  tracks:    5234

✅ artists and tracks have identical ID sets.
❌ The sets are not identical. Differences:

  IDs in artists or tracks but missing from reactions: [2, 8, 12, 13, 14, 23, 27, 30, 32, 36, 37, 40, 42, 50, 62, 63, 64, 70, 72, 76, 83, 84, 85, 91, 94, 106, 112, 114, 120, 121, 122, 136, 142, 171, 188, 198, 222, 252, 253, 272, 277, 286, 287, 311, 327, 331, 336, 346, 350, 387, 391, 397, 404, 405, 406, 413, 414, 416, 434, 439, 441, 467, 487, 506, 518, 526, 532, 549, 558, 559, 561, 569, 570, 572, 578, 582, 583, 594, 596, 600, 601, 625, 629, 630, 639, 653, 659, 664, 685, 689, 692, 697, 703, 707, 714, 715, 721, 734, 735, 736, 737, 744, 750, 761, 780, 781, 782, 801, 802, 805, 806, 807, 810, 818, 821, 822, 869, 878, 889, 890, 909, 921, 923, 928, 936, 938, 940, 959, 960, 966, 973, 976, 988, 992, 993, 994, 1004, 1037, 1042, 1058, 1060, 1064, 1078, 1082, 1087, 1088, 1089, 1121, 1122, 1123, 1141, 1159, 1168, 1182, 1183, 1202

# Real code, model training and evaluation

In [44]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict
import random
import os
import pickle
import json

SEED = 43
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
POSITIVE_SCORE_THRESHOLD = 1.0
TEST_PERCENT = 0.2
EMBEDDING_DIM = 200
BATCH_SIZE = 64
EPOCHS = 200
LR = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXPORT_DIR = 'model_params'

# -------------------------------------------------------------------
# 1. Load and prepare the new tables
# -------------------------------------------------------------------
def load_data(track_reactions_path, tracks_path, artists_path,
              artist_reactions_path, reaction_types_path):
    df_tr = pd.read_csv(track_reactions_path)
    df_tracks = pd.read_csv(tracks_path)
    df_artists = pd.read_csv(artists_path)
    df_ar = pd.read_csv(artist_reactions_path)
    df_rt = pd.read_csv(reaction_types_path)
    return df_tr, df_tracks, df_artists, df_ar, df_rt

def build_score_map(df_rt):
    return dict(zip(df_rt['id'], df_rt['score']))

# -------------------------------------------------------------------
# 2. Build user-artist matrix
# -------------------------------------------------------------------
def build_user_artist_matrix(df_tr, df_tracks, df_ar, score_map):
    track_to_artist = {}
    for _, row in df_tracks.iterrows():
        tid = row['id']
        artist_str = str(row['artists_id']) if pd.notna(row['artists_id']) else ''
        if artist_str:
            first_artist = int(artist_str.split(',')[0].strip())
            track_to_artist[tid] = first_artist
        else:
            track_to_artist[tid] = -1

    interactions = set()
    # track reactions
    for _, row in df_tr.iterrows():
        uid = row['user_id']
        tid = row['track_id']
        reaction_id = row['reaction_id']
        score = score_map.get(reaction_id, 0.0)
        if score > POSITIVE_SCORE_THRESHOLD:
            artist = track_to_artist.get(tid, -1)
            if artist != -1:
                interactions.add((uid, artist))

    # artist reactions
    for _, row in df_ar.iterrows():
        uid = row['user_id']
        artist_id = row['artist_id']
        reaction_id = row['reaction_id']
        score = score_map.get(reaction_id, 0.0)
        if score > POSITIVE_SCORE_THRESHOLD:
            interactions.add((uid, artist_id))

    # uploads (multiple uploaders per track)
    for _, row in df_tracks.iterrows():
        uploader_str = row['uploaded_by']
        if pd.isna(uploader_str):
            continue
        tid = row['id']
        artist = track_to_artist.get(tid, -1)
        if artist == -1:
            continue
        for uid_str in str(uploader_str).split(','):
            uid_str = uid_str.strip()
            if uid_str:
                uid = int(uid_str)
                interactions.add((uid, artist))

    # Encode
    users, artists = zip(*interactions)
    user_enc = LabelEncoder()
    artist_enc = LabelEncoder()
    user_idx = user_enc.fit_transform(users)
    artist_idx = artist_enc.fit_transform(artists)

    n_users = len(user_enc.classes_)
    n_artists = len(artist_enc.classes_)

    mat = csr_matrix((np.ones(len(interactions), dtype=np.float32),
                      (user_idx, artist_idx)),
                     shape=(n_users, n_artists))
    user_id_to_idx = {uid: i for i, uid in enumerate(user_enc.classes_)}
    return mat, user_id_to_idx, artist_enc, track_to_artist, user_enc

# -------------------------------------------------------------------
# 3. Build track-level split (unchanged from before, uses df_tr, score_map)
# -------------------------------------------------------------------
def build_track_split(df_tr, df_tracks, score_map, test_percent=0.2):
    interactions = []
    for _, row in df_tr.iterrows():
        uid = row['user_id']
        tid = row['track_id']
        reaction_id = row['reaction_id']
        score = score_map.get(reaction_id, 0.0)
        if score > POSITIVE_SCORE_THRESHOLD:
            interactions.append((uid, tid))

    user_enc = LabelEncoder()
    item_enc = LabelEncoder()
    u_idx = user_enc.fit_transform([x[0] for x in interactions])
    i_idx = item_enc.fit_transform([x[1] for x in interactions])

    n_users = len(user_enc.classes_)
    n_items = len(item_enc.classes_)

    mat = csr_matrix((np.ones(len(interactions)), (u_idx, i_idx)),
                     shape=(n_users, n_items))

    coo = mat.tocoo()
    np.random.seed(SEED)
    mask = np.random.rand(len(coo.data)) < (1 - test_percent)
    train = csr_matrix((coo.data[mask], (coo.row[mask], coo.col[mask])), shape=mat.shape)
    test  = csr_matrix((coo.data[~mask], (coo.row[~mask], coo.col[~mask])), shape=mat.shape)

    test_user_tracks = defaultdict(set)
    test_coo = test.tocoo()
    for u, i, v in zip(test_coo.row, test_coo.col, test_coo.data):
        if v > 0:
            uid = user_enc.classes_[u]
            tid = item_enc.classes_[i]
            test_user_tracks[uid].add(tid)

    track_pop = np.array(mat.sum(axis=0)).flatten()
    track_id_to_idx = {tid: i for i, tid in enumerate(item_enc.classes_)}

    seen_tracks = defaultdict(set)
    train_coo = train.tocoo()
    for u, i in zip(train_coo.row, train_coo.col):
        uid = user_enc.classes_[u]
        tid = item_enc.classes_[i]
        seen_tracks[uid].add(tid)

    return test_user_tracks, seen_tracks, track_pop, track_id_to_idx

# -------------------------------------------------------------------
# 4. BPR Dataset, Model, Loss (unchanged)
# -------------------------------------------------------------------
class BPRDataset(Dataset):
    def __init__(self, mat, num_neg=1):
        coo = mat.tocoo()
        self.users = torch.LongTensor(coo.row)
        self.pos_items = torch.LongTensor(coo.col)
        self.n_items = mat.shape[1]
        self.num_neg = num_neg

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos = self.pos_items[idx]
        negs = []
        while len(negs) < self.num_neg:
            neg = random.randint(0, self.n_items - 1)
            if neg != pos:
                negs.append(neg)
        return user, pos, torch.LongTensor(negs)

class MF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, user, item):
        u = self.user_emb(user)
        i = self.item_emb(item)
        return (u * i).sum(dim=-1)

def bpr_loss(model, user, pos, neg):
    pos_score = model(user, pos)
    neg_score = model(user, neg)
    diff = pos_score - neg_score
    return -torch.log(torch.sigmoid(diff) + 1e-10).mean()

# -------------------------------------------------------------------
# 5. Train user-artist model
# -------------------------------------------------------------------
def train_user_artist_model(mat, epochs=30):
    n_users, n_items = mat.shape
    dataset = BPRDataset(mat)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = MF(n_users, n_items, EMBEDDING_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for user, pos, negs in dataloader:
            user, pos, negs = user.to(DEVICE), pos.to(DEVICE), negs.squeeze(1).to(DEVICE)
            optimizer.zero_grad()
            loss = bpr_loss(model, user, pos, negs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 5 == 0:
            print(f"  epoch {epoch:02d}, loss={total_loss/len(dataloader):.4f}")

    model.eval()
    with torch.no_grad():
        user_emb = model.user_emb.weight.cpu().numpy()
        item_emb = model.item_emb.weight.cpu().numpy()
    return model, user_emb, item_emb

# -------------------------------------------------------------------
# 6. Evaluation (unchanged)
# -------------------------------------------------------------------
def evaluate_track_recall_and_precision(user_emb, item_emb, user_id_to_idx, artist_enc,
                                        track_to_artist, test_user_tracks, seen_tracks,
                                        track_pop, track_id_to_idx, top_k=10,
                                        n_artists=5, tracks_per_artist=5):
    artist_to_tracks = defaultdict(list)
    for tid, artist_id in track_to_artist.items():
        if artist_id in artist_enc.classes_ and tid in track_id_to_idx:
            aidx = artist_enc.transform([artist_id])[0]
            artist_to_tracks[aidx].append(tid)

    recalls = []
    precisions = []
    for uid, test_tids in test_user_tracks.items():
        if uid not in user_id_to_idx:
            continue
        u = user_id_to_idx[uid]
        scores = user_emb[u].dot(item_emb.T)
        top_artists = np.argpartition(scores, -n_artists)[-n_artists:]
        top_artists = top_artists[np.argsort(scores[top_artists])[::-1]]

        seen = seen_tracks.get(uid, set())
        candidates = []
        for aidx in top_artists:
            track_list = artist_to_tracks.get(aidx, [])
            track_list_sorted = sorted(track_list, key=lambda t: track_pop[track_id_to_idx[t]], reverse=True)
            for t in track_list_sorted[:tracks_per_artist]:
                if t not in seen:
                    candidates.append((t, track_pop[track_id_to_idx[t]]))
        uniq = {}
        for t, pop in candidates:
            uniq[t] = pop
        candidates = sorted(uniq.items(), key=lambda x: x[1], reverse=True)
        top_tracks = [t for t,_ in candidates[:top_k]]

        hit = len(set(top_tracks) & test_tids)
        recall = hit / len(test_tids) if test_tids else 0.0
        precision = hit / top_k
        recalls.append(recall)
        precisions.append(precision)

    return np.mean(recalls) if recalls else 0.0, np.mean(precisions) if precisions else 0.0

# -------------------------------------------------------------------
# 7. Export artifacts
# -------------------------------------------------------------------
def export_model(model, user_enc, artist_enc, track_to_artist,
                 track_pop, track_id_to_idx, user_emb, artist_emb,
                 export_dir=EXPORT_DIR):
    os.makedirs(export_dir, exist_ok=True)

    # Model state
    torch.save(model.state_dict(), os.path.join(export_dir, 'model_state.pt'))

    # Encoders and mappings
    with open(os.path.join(export_dir, 'user_enc.pkl'), 'wb') as f:
        pickle.dump(user_enc, f)
    with open(os.path.join(export_dir, 'artist_enc.pkl'), 'wb') as f:
        pickle.dump(artist_enc, f)
    with open(os.path.join(export_dir, 'track_to_artist.pkl'), 'wb') as f:
        pickle.dump(track_to_artist, f)
    with open(os.path.join(export_dir, 'track_id_to_idx.pkl'), 'wb') as f:
        pickle.dump(track_id_to_idx, f)

    # Arrays
    np.save(os.path.join(export_dir, 'track_pop.npy'), track_pop)
    np.save(os.path.join(export_dir, 'user_embeddings.npy'), user_emb)
    np.save(os.path.join(export_dir, 'artist_embeddings.npy'), artist_emb)

    # Config
    config = {
        'embedding_dim': EMBEDDING_DIM,
        'n_users': len(user_enc.classes_),
        'n_artists': len(artist_enc.classes_),
        'positive_score_threshold': POSITIVE_SCORE_THRESHOLD,
        'n_artists_rec': 5,          # number of top artists to consider
        'tracks_per_artist': 5,
        'top_k': 10
    }
    with open(os.path.join(export_dir, 'config.json'), 'w') as f:
        json.dump(config, f, indent=2)

    print(f"All model files exported to '{export_dir}'")

# -------------------------------------------------------------------
# 8. Main
# -------------------------------------------------------------------
def main():
    # Paths to CSV files (adjust as needed)
    df_tr, df_tracks, df_artists, df_ar, df_rt = load_data(
        'processed/track_reactions.csv',
        'processed/tracks.csv',
        'processed/artists.csv',
        'processed/artist_reactions.csv',
        'processed/reaction_types.csv'
    )

    score_map = build_score_map(df_rt)

    print("Building user-artist matrix...")
    ua_mat, user_id_to_idx, artist_enc, track_to_artist, user_enc = build_user_artist_matrix(
        df_tr, df_tracks, df_ar, score_map
    )
    print(f"User-Artist matrix: {ua_mat.shape[0]} users x {ua_mat.shape[1]} artists, {ua_mat.nnz} interactions")

    print("Building track test set...")
    test_user_tracks, seen_tracks, track_pop, track_id_to_idx = build_track_split(
        df_tr, df_tracks, score_map, TEST_PERCENT
    )

    print("Training user-artist model (PyTorch BPR)...")
    model, user_emb, artist_emb = train_user_artist_model(ua_mat, EPOCHS)

    print("Evaluating track recall & precision...")
    rec, prec = evaluate_track_recall_and_precision(
        user_emb, artist_emb, user_id_to_idx, artist_enc,
        track_to_artist, test_user_tracks, seen_tracks,
        track_pop, track_id_to_idx
    )
    print(f"Track Recall@10: {rec:.4f}   Precision@10: {prec:.4f}")

    # Export for server
    export_model(model, user_enc, artist_enc, track_to_artist,
                 track_pop, track_id_to_idx, user_emb, artist_emb)

if __name__ == "__main__":
    main()

Building user-artist matrix...
User-Artist matrix: 1284 users x 4996 artists, 59759 interactions
Building track test set...
Training user-artist model (PyTorch BPR)...
  epoch 05, loss=0.2361
  epoch 10, loss=0.1547
  epoch 15, loss=0.1322
  epoch 20, loss=0.1207
  epoch 25, loss=0.1159
  epoch 30, loss=0.1116
  epoch 35, loss=0.1075
  epoch 40, loss=0.1041
  epoch 45, loss=0.1057
  epoch 50, loss=0.1022
  epoch 55, loss=0.1043
  epoch 60, loss=0.0994
  epoch 65, loss=0.1008
  epoch 70, loss=0.0987
  epoch 75, loss=0.0992
  epoch 80, loss=0.0955
  epoch 85, loss=0.0965
  epoch 90, loss=0.0958
  epoch 95, loss=0.0946
  epoch 100, loss=0.0933
  epoch 105, loss=0.0925
  epoch 110, loss=0.0922
  epoch 115, loss=0.0926
  epoch 120, loss=0.0913
  epoch 125, loss=0.0900
  epoch 130, loss=0.0904
  epoch 135, loss=0.0887
  epoch 140, loss=0.0879
  epoch 145, loss=0.0896
  epoch 150, loss=0.0886
  epoch 155, loss=0.0871
  epoch 160, loss=0.0865
  epoch 165, loss=0.0866
  epoch 170, loss=0.0880
 